In [ ]:
from IPython import display

Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Master Course #24512

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# Exercise: Singular Value Decomposition (SVD) and Four Subspaces

## Objectives

- the four subspaces of a matrix are spanned by SVD's U and V matrices
- rank-1 matrix superposition

Before starting this notebook, (re)-consider [Matrix Fundamentals](MatrixFundamentals/MatrixFundamentals.ipynb) on how to actually calculate the SVD of matrix $\mathbf{A}$ from the eigenwert problem of $\mathbf{A}^\text{T}\mathbf{A}$.

## Special Python Packages

We use the functions from `scipy.linalg`, except `matrix_rank()` from `numpy.linalg`

## Highly Recommended Material
- Kenji Hiranabe's [The Art of Linear Algebra](https://github.com/kenjihiranabe/The-Art-of-Linear-Algebra)
- Gilbert Strang's
    - A 2020 Vision of Linear Algebra, Zoom Notes, 2021
    - A 2020 Vision of Linear Algebra, Slides, 2020
    - [Lecture Notes for Linear Algebra Sample Sections](https://math.mit.edu/~gs/LectureNotes/samples.pdf) and/or
    - [brilliant textbooks on linear algebra](https://math.mit.edu/~gs/) and/or
    - [video lecture in course 18.06](https://www.youtube.com/watch?v=TX_vooSnhm8) and/or
    - [video lecture in course 18.065](https://www.youtube.com/watch?v=rYz83XPxiZo)
- Steven L. Brunton's [video lecture(s) on SVD](https://www.youtube.com/watch?v=nbBvuuNVfco)

## Some Initial Python Stuff

In [ ]:
import numpy as np
from scipy.linalg import diagsvd, inv, norm, null_space, pinv, svd
from numpy.linalg import matrix_rank

np.set_printoptions(precision=3, floatmode="fixed", suppress=True)

rng = np.random.default_rng(42)

# we might convince ourselves that all works for complex data as well
# then the ^H operator (conj().T) needs to be used instead of just .T
use_complex = False

## Generate a Matrix

In [ ]:
# desired rank and dimensions
M = 7  # number of rows
N = 4  # number of cols
# set desired rank, here N-1 for a rank deficient matrix,
# i.e. neither full column nor full row rank
rank = min(M, N) - 1

if False:  # another small toy example
    M = 4  # number of rows
    N = 3  # number of cols
    rank = 2

print("desired rank of A:", rank)

if use_complex:
    dtype = "complex128"
    A = np.zeros([M, N], dtype=dtype)
    for i in range(rank):
        col = rng.integers(-3, 3, M) + 1j * rng.integers(-3, 3, M)
        row = rng.integers(-3, 3, N) + 1j * rng.integers(-3, 3, N)
        A += np.outer(col, row)  # superposition of rank-1 matrices
else:
    dtype = "int"
    A = np.zeros([M, N], dtype=dtype)
    for i in range(rank):
        col = rng.integers(-3, 3, M)
        row = rng.integers(-3, 3, N)
        A += np.outer(col, row)  # superposition of rank-1 matrices
        
# check if we produced desired matrix rank
print("        rank of A:", matrix_rank(A))
print("A =\n", A)

## SVD of Matrix A

In [ ]:
[U, s, Vh] = svd(A)
S = diagsvd(s, M, N)  # for full SVD the matrix S has same dim as A
V = Vh.conj().T

print("U =\n", U)
# number of non-zero sing values along diag must match rank:
print("\nnon-zero singular values: ", s[:rank], "\n")
print("S =\n", S)
print("V =\n", V)

## Four Subspaces in the SVD Domain

We denote 
- column space $C(\mathbf{A})$, aka *image*, *range* of $\mathbf{A}$, we might abbreviate this with CS
- left null space $N(\mathbf{A}^\mathrm{H})$, we might abbreviate this with LNS
- row space $C(\mathbf{A}^\mathrm{H})$, we might abbreviate this with RS
- null space $N(\mathbf{A})$, aka *kernel* of $\mathbf{A}$, we might abbreviate this with NS

The vectors of column space CS=$C(\mathbf{A})$ and left null space LNS=$N(\mathbf{A}^\mathrm{H})$ are related to the matrix $\mathbf{U}$ (matrix output).

The vectors of row space RS=$C(\mathbf{A}^\mathrm{H})$ and null space LNS=$N(\mathbf{A})$ are related to the matrix $\mathbf{V}$ (matrix input).

Since $\mathbf{U}$ and $\mathbf{V}$ are unitary matrices, they fulfill **orthonormality** and therefore the properties of orthogonal subspaces

$$C(\mathbf{A}) \perp N(\mathbf{A}^\mathrm{H})$$

$$C(\mathbf{A}^\mathrm{H}) \perp N(\mathbf{A})$$

immediately can be deduced. With our abbreviations this goes as

$$CS \perp LNS$$

$$RS \perp NS$$

In other words, since we know by the property of SVD, that the matrices $\mathbf{U}$ and $\mathbf{V}$ are orthonormal, we see that

- column space CS=$C(\mathbf{A})$ is orthogonal to left null space LNS=$N(\mathbf{A}^\mathrm{H})$, since both spaces are spanned by the dedicated vectors in $\mathbf{U}$
- row space RS=$C(\mathbf{A}^\mathrm{H})$ is orthogonal to null space NS=$N(\mathbf{A})$, since both spaces are spanned by the dedicated vectors in $\mathbf{V}$

# Big Picture of Four Subspaces / Forward Problem

<img src='slides/four_subspaces.png' alt="drawing" width="1000"/>

In [ ]:
# all stuff that is in matrix U (matrix output)
print("U =\n", U)

# column space C(A)
print("\nspan of the column space (orthogonal to left null space):")
print(U[:, :rank])

# left null space, if empty only the 0 vector is the left nullspace
print("span of the left null space (orthogonal to column space):")
print(U[:, rank:])

print("###")

# all stuff that is in matrix V (matrix input)
print("\nV =\n", V)

# row space
print("\nspan row space (orthogonal to null space):")
print(V[:, :rank])

# null space N(A), if empty only the 0 vector is the null space
print("span of null space (orthogonal to row space):")
print(V[:, rank:])

In [ ]:
# we use null_space() function

# this very often yields the same vectors as doing the SVD manually:
print("span of the null space: \n", null_space(A))

# this might be different from U[:,rank:], but spans the same space:
print("span of the left null space: \n", null_space(A.conj().T))
# we might want to check this (how should we do this actually?!)

## Pure Row Space --> Column Space

$$\mathbf{A}\,\mathbf{x}_{RS}=\mathbf{y}_{CS}$$

In [ ]:
# x as linear combination
# of the first two right singular vectors
# i.e. from the row space:
x = 1 * V[:, 0, None] + 2 * V[:, 1, None]
# let matrix A act on x
print(A @ x)
# result must be identical with the linear combination
# of the first two left singular vectors
# i.e. from the column space, weighted by their dedicated singular values and
# same weights 1,2 as above
print(1 * S[0, 0] * U[:, 0, None] + 2 * S[1, 1] * U[:, 1, None])

## Row Space + Null Space --> Column Space + Null Vector

$$\mathbf{A}\,\mathbf{x}_{RS+NS}=\mathbf{y}_{CS} + \mathbf{0}$$

In [ ]:
# x vector as a linear combination of row space and null space
x_r = 3 * V[:, 0, None]  # from row space
x_n = 4 * V[:, rank, None]  # from null space
print("x from pure null space must yield zero vector in the left nullspace:")
print("A @ x_n =\n", A @ x_n)
x = x_r + x_n  # linear combination
print("let matrix A act on x:")
print(A @ x)
print(
    "note: we transpose the following identical results to have convenient row display"
)
print((A @ x).T)
print((A @ x_r).T)
print((3 * S[0, 0] * U[:, 0, None]).T)

## Superposition of Rank-1 Matrices

Eckhart-Young theorem, cf. https://en.wikipedia.org/wiki/Low-rank_approximation


In [ ]:
Ar = np.zeros([M, N], dtype="complex128")
for r in range(rank):
    print("\nSum of", r + 1, "rank-1 matrices -> check (A-Ar)")
    Ar += S[r, r] * np.outer(U[:, r], V[:, r].conj())
    print(
        "Frobenius norm (i.e. root(sum squared sing val)): ",
        norm(A - Ar, "fro"),
    )
    print("Spectral norm / L2 norm (i.e. sigma1)", norm(A - Ar, 2))
    print("allclose(A, Ar):", np.allclose(A, Ar))

In the last case `Ar` is fully reconstructed from rank-1 matrices, yielding Frobenius and spectral norm for `A-Ar` of 0 (numerical precision does not give us zero). We can also check with allclose()

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- feel free to use the notebooks for your own purposes
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.